## Step 1: Install MCP

In [1]:
%pip install -qU "mcp[cli]"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: c:\Users\USER\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


## Step 2: Verify install

In [2]:
!python --version
!mcp version


Python 3.14.4
MCP version 1.27.1


## Step 3: server.py


In [3]:
%%writefile server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Demo")

@mcp.tool()
def add(a: int, b: int) -> int:
     """Return the sum of two integers."""
    return a + b

@mcp.resource("greeting://{name}")
def greet(name: str) -> str:
    """Return a greeting for the given name."""
    return f"Hello, {name}!"

if __name__ == "__main__":
    mcp.run(transport="stdio")

Overwriting server.py


## Step 4: client.py

In [4]:
%%writefile client.py
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server_params = StdioServerParameters(command="mcp", args=["run", "server.py"], env=None)


def extract_content(payload):
    """Best-effort to pull text from MCP responses."""
    if hasattr(payload, "contents"):
        contents = payload.contents
        if contents:
            first = contents[0]
            if hasattr(first, "text"):
                return first.text
            if isinstance(first, dict) and "text" in first:
                return first["text"]
            return str(first)
    if hasattr(payload, "content"):
        return payload.content
    return str(payload)


async def run():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # List resources (note: templates like greeting://{name} show up via list_resource_templates)
            resources = await session.list_resources()
            print("Resources:")
            for r in resources.resources:
                print(f"  - {r.uri}")

            templates = await session.list_resource_templates()
            print("Resource templates:")
            for t in templates.resourceTemplates:
                print(f"  - {t.uriTemplate}")

            # List tools
            tools = await session.list_tools()
            print("\nTools:")
            for t in tools.tools:
                print(f"  - {t.name}: {t.description}")

            # Read greeting://hello
            greeting = await session.read_resource("greeting://hello")
            print(f"\nGreeting: {extract_content(greeting)}")

            # Call add with a=1, b=7
            result = await session.call_tool("add", arguments={"a": 1, "b": 7})
            print(f"add(1, 7) = {extract_content(result)}")


if __name__ == "__main__":
    asyncio.run(run())

Overwriting client.py


In [9]:
!python client.py

Traceback (most recent call last):
  File "c:\Users\USER\code\GEN_AI_172\week18\day2\exerciseXP\client.py", line 2, in <module>
    from mcp import ClientSession, StdioServerParameters
ModuleNotFoundError: No module named 'mcp'
